# mmWave Radar Fall Detection — 3 Representation Comparative Benchmark

**Author**: Joey van der Poel  
**Affiliation**: University of Applied Sciences of Amsterdam  
**Date**: September 8, 2026  

---

## 🏆 Comparative Research Synthesis & Benchmark Results

This notebook evaluates the three radar data representations side-by-side to answer the core research questions outlined in the proposal (*'Comparing mmWave Radar Data Formats for Fall Detection'*):

| Representation | Representation Format | Feature Schema | Model Backbone | Core Theoretical Strength |
| :--- | :--- | :--- | :--- | :--- |
| **Representation 1** | **Micro-Doppler Spectrogram** | $64 \times 64$ Velocity-Time Heatmap | **ResNet-18** *(Kinematic)* | High sensitivity to sudden velocity bursts during fall collapse |
| **Representation 2** | **Orthogonal Projections** | Dual $64 \times 64$ ($XZ$ & $XY$) Grids | **ResNet-18** *(Spatial)* | Explicitly tracks vertical height drop ($Z$-axis) and posture expansion |
| **Representation 3** | **Native 3D Point Set** | Unordered Matrix $[5 \times 640]$ | **PointNet++** *(Geometric)* | Direct 3D spatial coordinate representation without grid quantization |


In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Load Pre-calculated Benchmark Results from JSON
results_path = Path("models/representation_benchmark_results.json")
if not results_path.exists():
    results_path = Path("../../models/representation_benchmark_results.json")
if not results_path.exists():
    results_path = Path("../models/representation_benchmark_results.json")
if results_path.exists():
    with open(results_path, "r") as f:
        res = json.load(f)
else:
    print("ERROR: Benchmark JSON file not found in models/")

# Format Side-by-Side Comparison Table
summary_data = []
for rep_name, m in res.items():
    summary_data.append({
        "Data Representation": rep_name.replace("_", " "),
        "Accuracy (%)": f"{m['accuracy']*100:.2f}%",
        "Recall (%)": f"{m['recall']*100:.2f}%",
        "Precision (%)": f"{m['precision']*100:.2f}%",
        "F1-Score": f"{m['f1_score']:.4f}",
        "ROC-AUC": f"{m['roc_auc']:.4f}",
        "ADL False Alarm Rate (FPR)": f"{m['fpr']*100:.2f}%"
    })

df_summary = pd.DataFrame(summary_data)
display(df_summary)


In [ ]:
# Overlayed ROC-AUC Comparison Curves
plt.figure(figsize=(9, 7))
colors = ["tab:red", "tab:blue", "tab:purple"]
labels = [
    "Rep 1: Spectrogram (ResNet-18 Kinematic)",
    "Rep 2: Orthogonal Projections (ResNet-18 Spatial)",
    "Rep 3: Native 3D PointSet (PointNet++ Geometric)"
]

for idx, (rep_key, m) in enumerate(res.items()):
    fpr, tpr, _ = roc_curve(m["test_targets"], m["test_probs"])
    plt.plot(fpr, tpr, color=colors[idx], lw=2.5, label=f"{labels[idx]} (AUC = {m['roc_auc']:.4f})")

plt.plot([0, 1], [0, 1], "k--", lw=1.5, label="Random Chance (AUC = 0.500)")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate (ADL False Alarms)", fontsize=12, fontweight="bold")
plt.ylabel("True Positive Rate (Fall Sensitivity/Recall)", fontsize=12, fontweight="bold")
plt.title("Overlayed ROC-AUC Curve Comparison across 3 Radar Data Formats", fontsize=14, fontweight="bold")
plt.legend(loc="lower right", fontsize=11)
plt.tight_layout()
plt.show()


In [ ]:
# Side-by-Side Confusion Matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cm_cmaps = ["Oranges", "Blues", "Purples"]

for idx, (rep_key, m) in enumerate(res.items()):
    cm = np.array(m["confusion_matrix"])
    sns.heatmap(cm, annot=True, fmt="d", ax=axes[idx], cmap=cm_cmaps[idx], cbar=False,
                xticklabels=["ADL", "Fall"], yticklabels=["ADL", "Fall"])
    axes[idx].set_title(labels[idx].split(":")[1], fontweight="bold")
    axes[idx].set_xlabel("Predicted Class")
    axes[idx].set_ylabel("True Class")

plt.suptitle("Side-by-Side Confusion Matrices (Fall vs. ADL False Positives)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
